# 🧠 MediFL — Federated Learning on NIH Chest X-Ray
### Sr. No. 29 · Privacy-Preserving Federated AI Framework for Healthcare

---
**What this notebook does:**
- Downloads NIH Chest X-Ray dataset from Kaggle
- Splits it into 3 hospital partitions (Mumbai 50%, Pune 30%, Delhi 20%)
- Trains a MobileNetV2 diagnostic model using **Federated Averaging (FedAvg)**
- Zero patient data shared between hospitals — only model weights
- Plots accuracy & loss curves per federated round
- Saves & downloads the trained global model

> ⚡ **Runtime:** Go to `Runtime → Change Runtime Type → T4 GPU` before running!

In [ ]:
# ============================================================
# CELL 1 — Install All Dependencies
# ============================================================
!pip install torch torchvision opacus opendatasets Pillow matplotlib scikit-learn gdown -q
print('✅ All packages installed successfully!')

In [ ]:
# ============================================================
# CELL 2 — Download NIH Chest X-Ray Dataset
# Uses opendatasets — will ask for Kaggle username + API key
#
# HOW TO GET YOUR KAGGLE KEY:
#   1. Go to https://www.kaggle.com → Account → API
#   2. Click 'Create New Token' → downloads kaggle.json
#   3. Open kaggle.json in Notepad
#   4. Copy the 'username' and 'key' values
#   5. Paste them when this cell prompts you
# ============================================================
import opendatasets as od
import os

# NIH Chest X-Ray SAMPLE (~2 GB) — from Problem Statement Sr.29
od.download(
    'https://www.kaggle.com/datasets/nih-chest-xrays/sample',
    data_dir='/content/'
)

DATA_DIR = '/content/sample'
print('✅ NIH Chest X-Ray dataset downloaded!')
!ls /content/sample/

In [ ]:
# ============================================================
# CELL 3 — Import All Libraries
# ============================================================
import os, copy, random, hashlib, json, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️  No GPU found. Go to Runtime → Change Runtime Type → T4 GPU')

In [ ]:
# ============================================================
# CELL 4 — Load & Explore the NIH Dataset CSV
# ============================================================
DATA_DIR = '/content/sample'

# Find CSV file
csv_file = None
for f in os.listdir(DATA_DIR):
    if f.endswith('.csv'):
        csv_file = os.path.join(DATA_DIR, f)
        break

if csv_file is None:
    # Try subdirectory
    for root, dirs, files in os.walk(DATA_DIR):
        for f in files:
            if f.endswith('.csv'):
                csv_file = os.path.join(root, f)
                break

df = pd.read_csv(csv_file)
print(f'✅ Dataset loaded: {len(df)} records')
print(f'   Columns: {df.columns.tolist()}')
print()
print('Top disease labels:')
print(df['Finding Labels'].value_counts().head(10))

In [ ]:
# ============================================================
# CELL 5 — Preprocess: Create Binary Labels & Find Images
# Normal = 0  |  Any Disease = 1
# ============================================================

# Binary label
def get_binary_label(label_str):
    return 0 if str(label_str).strip() == 'No Finding' else 1

df['binary_label'] = df['Finding Labels'].apply(get_binary_label)

# Find image folder
img_folder = None
for root, dirs, files in os.walk(DATA_DIR):
    png_files = [f for f in files if f.endswith('.png') or f.endswith('.jpg')]
    if len(png_files) > 10:
        img_folder = root
        break

print(f'📁 Image folder: {img_folder}')

# Filter to available images only
available_images = set(os.listdir(img_folder))
df = df[df['Image Index'].isin(available_images)].reset_index(drop=True)

print(f'✅ Available images: {len(df)}')
print(f'   Normal (0):  {(df["binary_label"]==0).sum()} images')
print(f'   Disease (1): {(df["binary_label"]==1).sum()} images')

In [ ]:
# ============================================================
# CELL 6 — NIH Dataset Class (PyTorch)
# ============================================================

class NIHChestDataset(Dataset):
    """NIH Chest X-Ray Dataset for MediFL hospital nodes."""

    def __init__(self, dataframe, img_folder, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_folder = img_folder
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Image Index']
        label    = self.df.loc[idx, 'binary_label']
        img_path = os.path.join(self.img_folder, img_name)
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

print('✅ NIHChestDataset class ready!')

In [ ]:
# ============================================================
# CELL 7 — Split Into 3 Hospital Partitions
# Simulates real-world federated hospital network
# ============================================================

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
total = len(df)

split_A = int(0.50 * total)  # Hospital A — Mumbai  → 50%
split_B = int(0.30 * total)  # Hospital B — Pune    → 30%
                              # Hospital C — Delhi   → 20% (remainder)

df_A = df.iloc[:split_A]
df_B = df.iloc[split_A : split_A + split_B]
df_C = df.iloc[split_A + split_B:]

print('✅ Dataset split into 3 hospital partitions:')
print(f'   🏥 Hospital A (Mumbai, MH)  → {len(df_A):>4} samples (50%) — NIH X-Rays')
print(f'   🏥 Hospital B (Pune, MH)    → {len(df_B):>4} samples (30%) — NIH X-Rays')
print(f'   🏥 Hospital C (Delhi, NCR)  → {len(df_C):>4} samples (20%) — NIH X-Rays')
print(f'   📊 Total: {total} records — Data NEVER leaves hospital boundaries!')

In [ ]:
# ============================================================
# CELL 8 — Diagnostic Model (MobileNetV2 Transfer Learning)
# Lightweight for federated training across nodes
# ============================================================

class DiagnosticModel(nn.Module):
    """
    MobileNetV2-based diagnostic model for NIH Chest X-Ray classification.
    Feature layers frozen — only classifier head is trained federatedly.
    """
    def __init__(self, num_classes=2):
        super(DiagnosticModel, self).__init__()
        backbone = models.mobilenet_v2(pretrained=True)
        # Freeze backbone — only fine-tune the classifier head
        for param in backbone.features.parameters():
            param.requires_grad = False
        # Custom classifier for binary chest diagnosis
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(backbone.last_channel, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

# Quick sanity check
test_m = DiagnosticModel().to(device)
out = test_m(torch.randn(2, 3, 224, 224).to(device))
print(f'✅ DiagnosticModel ready! Output shape: {out.shape}')
del test_m, out

In [ ]:
# ============================================================
# CELL 9 — Federated Learning Core Functions
# FedAvg (McMahan et al., 2017): W_global = Σ (n_k/N) * W_k
# ============================================================

def fedavg_aggregate(global_model, local_models, num_samples_list):
    """
    Federated Averaging Aggregation.
    ONLY model weights are aggregated — zero raw patient data shared.
    """
    total_samples = sum(num_samples_list)
    global_dict   = global_model.state_dict()

    for key in global_dict.keys():
        global_dict[key] = sum(
            (n / total_samples) * local_models[i].state_dict()[key].float()
            for i, n in enumerate(num_samples_list)
        )
    global_model.load_state_dict(global_dict)
    return global_model


def compute_model_hash(model):
    """SHA-256 hash of model weights for cryptographic audit trail."""
    weights_bytes = b''.join(
        v.cpu().numpy().tobytes()
        for v in model.state_dict().values()
    )
    return hashlib.sha256(weights_bytes).hexdigest()


def train_hospital_node(name, dataframe, img_folder, global_model,
                        epochs=2, lr=0.001, batch_size=32):
    """
    Local training at each hospital node.
    Data stays inside the hospital — only weights are returned.
    """
    print(f'   🏥 [{name}] Training on {len(dataframe)} local samples...')

    dataset = NIHChestDataset(dataframe, img_folder)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=True, num_workers=2, pin_memory=True)

    local_model = DiagnosticModel(num_classes=2).to(device)
    local_model.load_state_dict(copy.deepcopy(global_model.state_dict()))

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, local_model.parameters()), lr=lr
    )

    local_model.train()
    total_loss, correct, total = 0.0, 0, 0

    for epoch in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = local_model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            _, preds    = torch.max(outputs, 1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total * 100
    print(f'   ✅ [{name}] Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}%')
    return local_model, len(dataframe), avg_loss, accuracy


print('✅ Federated Learning functions ready!')

In [ ]:
# ============================================================
# CELL 10 — ▶️ RUN FEDERATED TRAINING (5 Rounds)
# ============================================================

# ── Config ─────────────────────────────────────────────────
FL_ROUNDS    = 5
LOCAL_EPOCHS = 2
LR           = 0.001
BATCH_SIZE   = 32

hospitals = [
    ('Hospital A (Mumbai)',  df_A),
    ('Hospital B (Pune)',    df_B),
    ('Hospital C (Delhi)',   df_C),
]

# ── Initialize Global Model ─────────────────────────────────
global_model = DiagnosticModel(num_classes=2).to(device)
print(f'✅ Global model initialized on {device}')
print(f'📊 Hospitals: {len(hospitals)} | Total Records: {total}')
print('🔒 Patient data will NOT leave hospital boundaries!\n')

# ── History ─────────────────────────────────────────────────
history = {'round': [], 'global_loss': [], 'global_acc': [],
           'A_loss': [], 'B_loss': [], 'C_loss': [],
           'A_acc':  [], 'B_acc':  [], 'C_acc':  []}

# ═══════════════════════════════════════════════════════════
#  FEDERATED TRAINING LOOP
# ═══════════════════════════════════════════════════════════
for fl_round in range(1, FL_ROUNDS + 1):
    print(f'\n{"="*58}')
    print(f'  🔄  FEDERATED ROUND  {fl_round} / {FL_ROUNDS}')
    print(f'{"="*58}')

    local_models, n_samples, losses, accs = [], [], [], []

    # Step 1 — Each hospital trains locally
    for hname, hdf in hospitals:
        lm, n, loss, acc = train_hospital_node(
            hname, hdf, img_folder, global_model,
            epochs=LOCAL_EPOCHS, lr=LR, batch_size=BATCH_SIZE
        )
        local_models.append(lm)
        n_samples.append(n)
        losses.append(loss)
        accs.append(acc)

    # Step 2 — Central aggregator: FedAvg
    print(f'\n   🧮 [AGGREGATOR] FedAvg across {len(hospitals)} hospital nodes...')
    global_model = fedavg_aggregate(global_model, local_models, n_samples)

    # Step 3 — Global weighted metrics
    N = sum(n_samples)
    g_loss = sum(l*n for l,n in zip(losses, n_samples)) / N
    g_acc  = sum(a*n for a,n in zip(accs,   n_samples)) / N
    h      = compute_model_hash(global_model)

    print(f'\n   📊 [GLOBAL MODEL] Round {fl_round} Results:')
    print(f'      Weighted Loss:     {g_loss:.4f}')
    print(f'      Weighted Accuracy: {g_acc:.2f}%')
    print(f'      SHA-256 Hash:      {h[:28]}...')

    # Store history
    history['round'].append(fl_round)
    history['global_loss'].append(g_loss)
    history['global_acc'].append(g_acc)
    history['A_loss'].append(losses[0]);  history['A_acc'].append(accs[0])
    history['B_loss'].append(losses[1]);  history['B_acc'].append(accs[1])
    history['C_loss'].append(losses[2]);  history['C_acc'].append(accs[2])

print(f'\n{"="*58}')
print(f'  ✅  FEDERATED TRAINING COMPLETE!')
print(f'  Final Global Accuracy : {history["global_acc"][-1]:.2f}%')
print(f'  Final Global Loss     : {history["global_loss"][-1]:.4f}')
print(f'{"="*58}')

In [ ]:
# ============================================================
# CELL 11 — Plot Results (Dashboard Style)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f172a')
fig.suptitle('MediFL — NIH Chest X-Ray | Federated Learning Results (Sr. No. 29)',
             color='white', fontsize=14, fontweight='bold', y=1.02)

rounds = history['round']
colors = {'bg': '#1e293b', 'grid': 'rgba(255,255,255,0.05)',
          'green': '#10b981', 'purple': '#6366f1',
          'blue': '#0ea5e9', 'amber': '#f59e0b'}

for ax in axes:
    ax.set_facecolor(colors['bg'])
    ax.tick_params(colors='#94a3b8')
    ax.xaxis.label.set_color('#94a3b8')
    ax.yaxis.label.set_color('#94a3b8')
    ax.title.set_color('white')
    for sp in ax.spines.values(): sp.set_edgecolor('#334155')
    ax.grid(alpha=0.12, color='white')

# Plot 1: Global Accuracy
axes[0].plot(rounds, history['global_acc'], 'o-',
             color=colors['green'], linewidth=2.5, markersize=8)
axes[0].fill_between(rounds, history['global_acc'], alpha=0.15, color=colors['green'])
axes[0].set_title('🌐 Global Model Accuracy', fontweight='bold')
axes[0].set_xlabel('Federated Round'); axes[0].set_ylabel('Accuracy (%)')

# Plot 2: Global Loss
axes[1].plot(rounds, history['global_loss'], 'o-',
             color=colors['purple'], linewidth=2.5, markersize=8)
axes[1].fill_between(rounds, history['global_loss'], alpha=0.15, color=colors['purple'])
axes[1].set_title('📉 Global Weighted Loss', fontweight='bold')
axes[1].set_xlabel('Federated Round'); axes[1].set_ylabel('Loss')

# Plot 3: Per-Hospital Loss
axes[2].plot(rounds, history['A_loss'], 'o-', color=colors['purple'],
             linewidth=2, markersize=7, label='Hospital A (Mumbai)')
axes[2].plot(rounds, history['B_loss'], 's-', color=colors['blue'],
             linewidth=2, markersize=7, label='Hospital B (Pune)')
axes[2].plot(rounds, history['C_loss'], '^-', color=colors['amber'],
             linewidth=2, markersize=7, label='Hospital C (Delhi)')
axes[2].plot(rounds, history['global_loss'], 'd--', color=colors['green'],
             linewidth=2.5, markersize=8, label='Global Federated')
axes[2].set_title('🏥 Per-Hospital Loss Comparison', fontweight='bold')
axes[2].set_xlabel('Federated Round'); axes[2].set_ylabel('Loss')
axes[2].legend(facecolor='#0f172a', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('medifl_results.png', dpi=150,
            bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Results chart saved as medifl_results.png')

In [ ]:
# ============================================================
# CELL 12 — Print Summary Table
# ============================================================

print('\n' + '='*60)
print('  MediFL — TRAINING SUMMARY TABLE')
print('='*60)
print(f'{"Round":<8} {"Global Acc":>12} {"Global Loss":>13} {"A Loss":>10} {"B Loss":>10} {"C Loss":>10}')
print('-'*60)
for i, r in enumerate(history['round']):
    print(f'{r:<8} {history["global_acc"][i]:>11.2f}% '
          f'{history["global_loss"][i]:>13.4f} '
          f'{history["A_loss"][i]:>10.4f} '
          f'{history["B_loss"][i]:>10.4f} '
          f'{history["C_loss"][i]:>10.4f}')
print('='*60)
print(f'\n  Dataset  : NIH Chest X-Ray (Public Medical Imaging)')
print(f'  Strategy : Federated Averaging — FedAvg (McMahan 2017)')
print(f'  Hospitals: 3 (Mumbai 50% | Pune 30% | Delhi 20%)')
print(f'  Privacy  : Zero patient data crossed hospital boundaries')
print('='*60)

In [ ]:
# ============================================================
# CELL 13 — Save & Download Trained Model + Results
# ============================================================
import os
from google.colab import files

os.makedirs('/content/medifl_output', exist_ok=True)

# Save global model
model_path = '/content/medifl_output/medifl_global_model.pth'
torch.save(global_model.state_dict(), model_path)
print(f'✅ Global model saved → {model_path}')

# Save training history
hist_path = '/content/medifl_output/training_history.json'
with open(hist_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'✅ Training history saved → {hist_path}')

# Download everything to your laptop
print('\n📥 Downloading files to your laptop...')
files.download(model_path)
files.download(hist_path)
files.download('medifl_results.png')
print('✅ All files downloaded!')
print('\n  Files you received:')
print('  📦 medifl_global_model.pth   → Trained model weights')
print('  📊 training_history.json     → Round-by-round metrics')
print('  🖼️  medifl_results.png        → Charts for your report/viva')